# ODS v2: Home quotes (HFQ

Builds 6 ods tables in `ods_v2`: `QuoteHome`, `PolicyHomeQsEvent`, `QuoteHomePrice`, `QuoteHomePaymentRequest`, `QuoteHomePaymentResponse`, `QuoteHomeSale`.

Needs these notebooks to have succeeded first: 20_Quote.

Generated by `ods_build/v2/build_notebook.py` from `entities/*.yaml`: one SQL statement per cell.

**If a cell fails**

1. Read the failing cell's header: what it builds, what it reads, its grain, key and dedupe rule.
2. Fix the cause. Most failures are a source change (a renamed or missing column) or a broken key (duplicates).
   The checks cell at the end names the table and the likely cause.
3. Re-run from the failed cell: every cell rebuilds its table completely (`create or replace`), so re-running is
   always safe. After a session restart, run the first three cells again first (parameters, setup, start log).
4. **Copy the fix back into the config** (`ods_build/v2/entities/*.yaml`) and regenerate. This notebook is generated:
   a fix made only here is lost at the next generation (the drift check lists such cells).


In [ ]:
%%pyspark
# Parameters (set by ODS_v2_Run; leave blank when running by hand)
batch_id   = ""
source_set = ""


In [ ]:
%%pyspark
%run ODS_v2_00_Setup

In [ ]:
%%pyspark
ods_run_log("22_Quote_Home", "started")


## QuoteHome


In [ ]:
/******************************************************************************************************/
/*  Name        : ${ods.stg}.odsv2_quote_home_cc_detail                                                 */
/*  Description : Home QS quote detail, latest transaction per quote id, with specified items and price */
/*              :  range.                                                                               */
/*              :                                                                                       */
/*              : Reads: dlk.ext_home_qs_policydetails, dlk.ext_home_qs_riskdetails, dlk.ext_home_qs_sp */
/*              : ecifieditemsdetails, dlk.ext_home_qs_quotedetails                                     */
/******************************************************************************************************/
create or replace table ${ods.stg}.odsv2_quote_home_cc_detail as
select  *
from    (
        select  upper(a.PolicyId)                                   as Ref,
                a.TransactionDate                                   as QuotationDate,
                a.BusinessProcess                                   as QuoteType,
                cast(a.EffectiveStartDate as date)                  as CoverStartDate,
                cast(add_months(a.EffectiveStartDate, 12) as date)  as CoverEndDate,
                b.ResidenceDate                                     as HomeResidenceStartDate,
                cast(b.OccupiedDuringDate as string)                as OccupiedDuringDay,
                cast(b.NeigbourhoodWatch as string)                 as NeighbourhoodWatch,
                b.ResidenceType,
                cast(b.IsLandlord as string)                        as IsLandlord,
                b.NumberofTimesLetinaYear                           as NumberOfTimesLetInYear,
                b.RoofConstructionType,
                b.NumberofSmokeAlarms                               as NumberOfSmokeAlarms,
                b.HeatingType,
                cast(b.HasBasement as string)                       as HasBasement,
                b.NumberofBedrooms                                  as NumberOfBedrooms,
                b.NumberofBathrooms                                 as NumberOfBathrooms,
                b.PropertyType,
                b.ClaimedYearsNCD,
                c.ItemSumInsuredRequired,
                c.InsuredItemsCount,
                d.MinPremium,
                d.MaxPremium,
                row_number() over (partition by upper(a.PolicyId) order by a.TransactionDate desc, a.`$ExtractDateTime` desc) as rn
        from    ${ods.dlk}.ext_home_qs_policydetails    a
        left join ${ods.dlk}.ext_home_qs_riskdetails    b   on b.HomeRiskId = upper(a.RiskId)
        left join (
                select  r.HomeCoverId, sum(i.ItemSumInsuredRequired) as ItemSumInsuredRequired, count(*) as InsuredItemsCount
                from    ${ods.dlk}.ext_home_qs_riskdetails              r
                join    ${ods.dlk}.ext_home_qs_specifieditemsdetails    i   on i.HomeCoverId = r.HomeCoverId
                where   i.ItemType <> 'Unspecified'
                group by r.HomeCoverId
                ) c     on c.HomeCoverId = b.HomeCoverId
        left join (
                select  upper(p.PolicyId) as Ref, min(q.TotalPayable) as MinPremium, max(q.TotalPayable) as MaxPremium
                from    ${ods.dlk}.ext_home_qs_policydetails p
                join    ${ods.dlk}.ext_home_qs_quotedetails  q   on q.HomeRiskId = upper(p.RiskId)
                where   q.Outcome = 'Premium Returned' and q.TotalPayable > 0
                group by upper(p.PolicyId)
                ) d     on d.Ref = upper(a.PolicyId)
        ) x
where   rn = 1

In [ ]:
/******************************************************************************************************/
/*  Name        : ${ods.stg}.odsv2_quote_home_web_detail                                                */
/*  Description : HFQ quote detail, latest version per quote reference, journey fields and hashed conta */
/*              : ct keys.                                                                              */
/*              :                                                                                       */
/*              : Reads: dlk.hfq_quotedetails                                                           */
/******************************************************************************************************/
create or replace table ${ods.stg}.odsv2_quote_home_web_detail as
select  *
from    (
        select  nullif(upper(trim(a.QuoteCodeReference)), '')                          as Ref,
                a.CreatedOn                                         as QuotationDate,
                a.UpdatedOn,
                'New Business'                                      as QuoteType,
                cast(a.policyDetails_startDate as date)             as CoverStartDate,
                cast(add_months(a.policyDetails_startDate, 12) as date) as CoverEndDate,
                cast(a.residencyDetails_isOccupiedDuringTheDay as string) as OccupiedDuringDay,
                cast(a.propertyDetails_securityDetails_hasNeighbourhoodWatch as string) as NeighbourhoodWatch,
                case a.residencyDetails_typeOf
                     when 'HolidayHomeFamilyUse' then 'Holiday Home - Family Use'
                     when 'HolidayHomeLetOut'    then 'Holiday Home - Let Out'
                     when 'OwnerOccupied'        then 'Owner Occupied'
                     when 'RentedCouple'         then 'Rented - Couple'
                     when 'RentedFamily'         then 'Rented - Family'
                     when 'RentedFivePeople'     then 'Rented - 5 People'
                     when 'RentedFourPeople'     then 'Rented - 4 People'
                     when 'RentedOnePerson'      then 'Rented - 1 Person'
                     when 'RentedSixPeople'      then 'Rented - 6 People'
                     when 'RentedStudents'       then 'Rented - Students'
                     when 'RentedTenPeople'      then 'Rented - 10 People'
                     when 'RentedThreePeople'    then 'Rented - 3 People'
                     when 'RentedTwoPeople'      then 'Rented - 2 People'
                     when 'Unoccupied'           then 'Unoccupied'
                     else 'Unknown' end                             as ResidenceType,
                cast(a.ResidencyDetails_IsLandlord as string)       as IsLandlord,
                case when a.propertyDetails_roofConstructionDetails_is100PercentStandardMaterial = 1 then 'Standard'
                     when a.PropertyDetails_RoofConstructionDetails_TypeOfNonStandardMaterialUsed is not null
                          then a.PropertyDetails_RoofConstructionDetails_TypeOfNonStandardMaterialUsed
                     else 'NotListed' end                           as RoofConstructionType,
                a.propertyDetails_securityDetails_numberOfSmokeAlarms as NumberOfSmokeAlarms,
                a.propertyDetails_typeOfHeatingSystem               as HeatingType,
                cast(a.propertyDetails_hasABasement as string)      as HasBasement,
                a.propertyDetails_numberOfBedrooms                  as NumberOfBedrooms,
                a.propertyDetails_numberOfBathrooms                 as NumberOfBathrooms,
                a.propertyDetails_typeOf                            as PropertyType,
                a.coverDetails_contentsCoverAmount                  as ItemSumInsuredRequired,
                try_cast(a.RetrieveCount as int)                    as RetrieveCount,
                a.MaxStep,
                a.LastStep,
                a.proposer_previousPolicyDetails_ExpiryDate         as PreviousPolicyExpiryDate,
                a.ImpervaIsBot,
                a.ImpervaSuspectedBot,
                ods_hash_email(a.proposer_email)                        as EmailKey,
                ods_hash_phone(a.proposer_mobilePhoneNumber)            as MobileKey,
                ods_eircode_area(a.propertyDetails_address_eircode)     as EircodeArea,
                ods_hash_id(concat_ws(' ', a.propertyDetails_Address_Line1, a.propertyDetails_Address_Line2), '') as AddressKey,
                row_number() over (partition by upper(trim(a.QuoteCodeReference))
                                   order by a.UpdatedOn desc, a.InsertDateTime desc) as rn
        from    ${ods.dlk}.hfq_quotedetails a
        where   nullif(trim(a.QuoteCodeReference), '') is not null
        ) x
where   rn = 1

In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.QuoteHome                                                                    */
/*  Description : Home quote detail from the web (HFQ, 3) and the contact centre (Home QS, 4), as noteb */
/*              : ook 02, one row per quote                                                             */
/*              : (the web side keeps its latest version). Web-only journey fields (retrievals, steps,  */
/*              : bot flags) and hashed contact                                                         */
/*              : keys are added; the Eircode is reduced to its routing key.                            */
/*              :                                                                                       */
/*              : Grain: one row per home quote                                                         */
/*              : Business key: SourceSystemId + SourceSystemQuoteReference                             */
/*              : History: current   Cut-off: none                                                      */
/*              : Source: dlk.hfq_quotedetails (+ hfq_sales, hfq_response_quotes), dlk.ext_home_qs_* (c */
/*              : ontact centre)                                                                        */
/*              : Reads: stg.odsv2_quote_home_cc_detail, ods_v2.Quote, stg.odsv2_quote_home_web_detail  */
/******************************************************************************************************/
create or replace table ${ods.target}.QuoteHome as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  q.QuoteId, 4 as SourceSystemId, c.Ref as SourceSystemQuoteReference, q.SourcePolicyCode, q.PolicyId,
                c.QuotationDate, cast(null as timestamp) as UpdatedOn, c.QuoteType, c.CoverStartDate, c.CoverEndDate,
                c.HomeResidenceStartDate, c.OccupiedDuringDay, c.NeighbourhoodWatch, c.ResidenceType, c.IsLandlord,
                c.NumberOfTimesLetInYear, c.RoofConstructionType, c.NumberOfSmokeAlarms, c.HeatingType, c.HasBasement,
                c.NumberOfBedrooms, c.NumberOfBathrooms, cast(c.PropertyType as string) as PropertyType, c.ClaimedYearsNCD,
                c.ItemSumInsuredRequired, c.InsuredItemsCount, c.MinPremium, c.MaxPremium,
                cast(null as int) as RetrieveCount, cast(null as string) as MaxStep, cast(null as string) as LastStep,
                cast(null as date) as PreviousPolicyExpiryDate, cast(null as string) as ImpervaIsBot,
                cast(null as string) as ImpervaSuspectedBot, cast(null as string) as EmailKey, cast(null as string) as MobileKey,
                cast(null as string) as EircodeArea, cast(null as string) as AddressKey
        from    ${ods.stg}.odsv2_quote_home_cc_detail c
        join    ${ods.target}.Quote q on q.SourceSystemId = 4 and q.SourceSystemQuoteReference = c.Ref
        union all
        select  q.QuoteId, 3, w.Ref, q.SourcePolicyCode, q.PolicyId,
                w.QuotationDate, w.UpdatedOn, w.QuoteType, w.CoverStartDate, w.CoverEndDate,
                cast(null as date), w.OccupiedDuringDay, w.NeighbourhoodWatch, w.ResidenceType, w.IsLandlord,
                cast(null as int), w.RoofConstructionType, w.NumberOfSmokeAlarms, w.HeatingType, w.HasBasement,
                w.NumberOfBedrooms, w.NumberOfBathrooms, cast(w.PropertyType as string), cast(null as int),
                w.ItemSumInsuredRequired, cast(null as bigint), q.QuotedPremium, cast(null as decimal(18,2)),
                w.RetrieveCount, cast(w.MaxStep as string), cast(w.LastStep as string),
                cast(w.PreviousPolicyExpiryDate as date), cast(w.ImpervaIsBot as string),
                cast(w.ImpervaSuspectedBot as string), w.EmailKey, w.MobileKey,
                w.EircodeArea, w.AddressKey
        from    ${ods.stg}.odsv2_quote_home_web_detail w
        join    ${ods.target}.Quote q on q.SourceSystemId = 3 and q.SourceSystemQuoteReference = w.Ref
        ) x

## PolicyHomeQsEvent


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.PolicyHomeQsEvent                                                            */
/*  Description : Every Home QS (contact-centre home) policy transaction: business process, event type  */
/*              : and dates, per risk. Feeds                                                            */
/*              : the home renewal and home claim events (notebook 04 cells 197-199, 273-275), which jo */
/*              : in it to the claims on the                                                            */
/*              : risk id. HomeQsPolicyId is the Home QS quote / policy id (= Quote.SourceSystemQuoteRe */
/*              : ference for system 4).                                                                */
/*              :                                                                                       */
/*              : Grain: one row per Home QS transaction (risk id)                                      */
/*              : Business key: HomeRiskId                                                              */
/*              : History: event   Cut-off: TransactionDate                                             */
/*              : Source: dlk.ext_home_qs_policydetails                                                 */
/*              : Reads: dlk.ext_home_qs_policydetails, ods_v2.Quote, ods_v2.Policy                     */
/******************************************************************************************************/
create or replace table ${ods.target}.PolicyHomeQsEvent as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  upper(p.RiskId)                                                 as HomeRiskId,
                upper(p.PolicyId)                                               as HomeQsPolicyId,
                q.QuoteId,
                pl.PolicyId,
                nullif(upper(trim(p.PolicyCode)), '')                                              as PolicyCode,
                p.BusinessProcess,
                p.EventType,
                p.TransactionDate,
                p.EffectiveStartDate,
                p.EffectiveEndDate,
                p.RenewalDate,
                p.InceptionDate,
                p.WithBrokerFromDate,
                nullif(upper(trim(p.PolicyNumber)), '')                                            as PolicyNumber,
                ods_redact(nullif(upper(trim(p.PolicyReference)), ''))                                 as PolicyReference,         -- typed in: 39 emails, some mobiles
                p.InsuranceCompany,
                p.BrokerCompany,
                p.AgencyName,
                p.`$ExtractDateTime`                                            as ExtractDateTime
        from    ${ods.dlk}.ext_home_qs_policydetails    p
        left join ${ods.target}.Quote               q   on q.SourceSystemId = 4 and q.SourceSystemQuoteReference = upper(p.PolicyId)
        left join ${ods.target}.Policy              pl  on pl.PolicyCode = upper(trim(p.PolicyCode))
        where   p.TransactionDate >= date '${ods.cutoff_date}'
        ) x

## QuoteHomePrice


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.QuoteHomePrice                                                               */
/*  Description : Each insurer / scheme price returned to an HFQ home quote, with excesses, levy and ou */
/*              : tcome.                                                                                */
/*              :                                                                                       */
/*              : Grain: one row per HFQ quote response row (insurer / scheme price)                    */
/*              : Business key: ResponseQuoteId                                                         */
/*              : History: event   Cut-off: DateCreated                                                 */
/*              : Source: dlk.hfq_response_quotes                                                       */
/*              : Reads: dlk.hfq_response_quotes, ods_v2.Quote                                          */
/******************************************************************************************************/
create or replace table ${ods.target}.QuoteHomePrice as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  r.unq_id                                                        as ResponseQuoteId,
                q.QuoteId,
                nullif(upper(trim(r.QuoteCodeReference)), '')                                      as QuoteCodeReference,
                nullif(upper(trim(r.PolicyCode)), '')                                              as PolicyCode,
                r.Quotes_InsurerName                                            as InsurerName,
                r.Quotes_SchemeName                                             as SchemeName,
                r.Quotes_SchemeCode                                             as SchemeCode,
                r.Quotes_SchemeCategory                                         as SchemeCategory,
                r.Quotes_Premium                                                as Premium,
                r.Quotes_TotalPayable                                           as TotalPayable,
                r.Quotes_TotalExcess                                            as TotalExcess,
                r.Quotes_CompulsoryExcess                                       as CompulsoryExcess,
                r.Quotes_LevyAmount                                             as LevyAmount,
                r.Quotes_Outcome                                                as Outcome,
                r.Quotes_IsIndicative                                           as IsIndicative,
                r.Quotes_DateCreated                                            as DateCreated,
                r.Quotes_DateValidTo                                            as DateValidTo,
                r.Quotes_CoverEffectiveFrom                                     as CoverEffectiveFrom,
                r.Quotes_CoverEffectiveTo                                       as CoverEffectiveTo
        from    ${ods.dlk}.hfq_response_quotes r
        left join ${ods.target}.Quote q on q.SourceSystemId = 3 and q.SourceSystemQuoteReference = upper(trim(r.QuoteCodeReference))
        where   r.Quotes_DateCreated >= date '${ods.cutoff_date}'
        ) x

## QuoteHomePaymentRequest


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.QuoteHomePaymentRequest                                                      */
/*  Description : HFQ payment requests sent to the payment provider. Payer name and billing address are */
/*              :  not taken; the payer email                                                           */
/*              : is a salted hash.                                                                     */
/*              :                                                                                       */
/*              : Grain: one row per HFQ payment request (global payment reference)                     */
/*              : Business key: GlobalPaymentReference                                                  */
/*              : History: event   Cut-off: InsertDateTime                                              */
/*              : Source: dlk.hfq_paymentrequest_request                                                */
/*              : Reads: dlk.hfq_paymentrequest_request, ods_v2.Quote                                   */
/******************************************************************************************************/
create or replace table ${ods.target}.QuoteHomePaymentRequest as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  x.GlobalPaymentReference, q.QuoteId, x.QuoteReference, x.Reference, x.TemporaryQuoteId, x.CacheId, x.Amount,
                x.DepositAmount, x.MonthlyAmount, x.IsFullPayment, x.PolicyStartDate, x.SchemeCode, x.BrokerID, x.RequestType,
                x.InsertDateTime, x.PayerEmailKey
        from    (
                select  globalPayment_reference                                 as GlobalPaymentReference,
                        nullif(upper(trim(QuoteReference)), '')                                    as QuoteReference,
                        reference                                               as Reference,
                        TemporaryQuoteId,
                        CacheId,
                        amount                                                  as Amount,
                        depositAmount                                           as DepositAmount,
                        monthlyAmount                                           as MonthlyAmount,
                        isFullPayment                                           as IsFullPayment,
                        policyStartDate                                         as PolicyStartDate,
                        schemeCode                                              as SchemeCode,
                        BrokerID,
                        type                                                    as RequestType,
                        InsertDateTime,
                        ods_hash_email(payer_email)                                 as PayerEmailKey,
                        row_number() over (partition by globalPayment_reference order by InsertDateTime desc) as rn
                from    ${ods.dlk}.hfq_paymentrequest_request
                where   nullif(trim(globalPayment_reference), '') is not null
                and     InsertDateTime >= date '${ods.cutoff_date}'
                ) x
        left join ${ods.target}.Quote q on q.SourceSystemId = 3 and q.SourceSystemQuoteReference = x.QuoteReference
        where   x.rn = 1
        ) x

## QuoteHomePaymentResponse


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.QuoteHomePaymentResponse                                                     */
/*  Description : HFQ payment provider responses: status, result code and amount per transaction. Card  */
/*              : details are not taken.                                                                */
/*              :                                                                                       */
/*              : Grain: one row per HFQ payment response (provider transaction id)                     */
/*              : Business key: PaymentResponseId                                                       */
/*              : History: event   Cut-off: InsertDateTime                                              */
/*              : Source: dlk.hfq_paymentresponse_response                                              */
/*              : Reads: dlk.hfq_paymentresponse_response, ods_v2.Quote                                 */
/******************************************************************************************************/
create or replace table ${ods.target}.QuoteHomePaymentResponse as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  x.PaymentResponseId, q.QuoteId, x.QuoteReference, x.Reference, x.TimeCreated, x.ResponseType, x.Status, x.Channel,
                x.Amount, x.Currency, x.PaymentMethodResult, x.PaymentMethodMessage, x.ActionType, x.ActionResultCode,
                x.ActionTimeCreated, x.InsertDateTime
        from    (
                select  id                                                      as PaymentResponseId,
                        nullif(upper(trim(QuoteReference)), '')                                    as QuoteReference,
                        reference                                               as Reference,
                        time_created                                            as TimeCreated,
                        type                                                    as ResponseType,
                        status                                                  as Status,
                        channel                                                 as Channel,
                        amount                                                  as Amount,
                        currency                                                as Currency,
                        payment_method_result                                   as PaymentMethodResult,
                        payment_method_message                                  as PaymentMethodMessage,
                        action_type                                             as ActionType,
                        action_result_code                                      as ActionResultCode,
                        action_time_created                                     as ActionTimeCreated,
                        InsertDateTime,
                        row_number() over (partition by id order by InsertDateTime desc) as rn
                from    ${ods.dlk}.hfq_paymentresponse_response
                where   nullif(trim(id), '') is not null
                and     InsertDateTime >= date '${ods.cutoff_date}'
                ) x
        left join ${ods.target}.Quote q on q.SourceSystemId = 3 and q.SourceSystemQuoteReference = x.QuoteReference
        where   x.rn = 1
        ) x

## QuoteHomeSale


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.QuoteHomeSale                                                                */
/*  Description : HFQ web sales feed: the quote reference, the code it was sold under, sale date and st */
/*              : atus (Status 1 = sold, as                                                             */
/*              : notebook 02; other values undocumented). Latest load per sale id: the feed reloads re */
/*              : cords, and the earlier copies                                                         */
/*              : carry no status. PolicyCode is as the feed holds it and is empty for some records (a  */
/*              : later reload may fill it);                                                            */
/*              : linking a sale to a policy is derived in ads (warehouse HEQ matching: left(PolicyCode */
/*              : , 6) as client code, then                                                             */
/*              : email / mobile / address), not here. Payer and payment JSON are not taken.            */
/*              :                                                                                       */
/*              : Grain: one row per HFQ sale record id (latest load)                                   */
/*              : Business key: SaleRecordId                                                            */
/*              : History: event (no cut-off; small)   Cut-off: none                                    */
/*              : Source: dlk.hfq_sales                                                                 */
/*              : Reads: dlk.hfq_sales, ods_v2.Quote, ods_v2.Policy                                     */
/******************************************************************************************************/
create or replace table ${ods.target}.QuoteHomeSale as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  x.SaleRecordId, q.QuoteId, p.PolicyId, x.QuoteCodeReference, x.PolicyCode, x.SourceQuoteId, x.TemporaryQuoteId,
                x.CacheID, x.GlobalPaymentReference, x.SchemeCode, x.Status, x.SaleDate, x.InsertDateTime
        from    (
                select  id                                                      as SaleRecordId,
                        nullif(upper(trim(QuoteCodeReference)), '')                                as QuoteCodeReference,
                        nullif(upper(trim(PolicyCode)), '')                                        as PolicyCode,
                        quoteid                                                 as SourceQuoteId,
                        TemporaryQuoteId,
                        CacheID,
                        GlobalPaymentReference,
                        SchemeCode,
                        Status,
                        SaleDate,
                        InsertDateTime,
                        row_number() over (partition by id order by InsertDateTime desc, Status desc) as rn
                from    ${ods.dlk}.hfq_sales
                where   nullif(trim(id), '') is not null
                ) x
        left join ${ods.target}.Quote   q   on q.SourceSystemId = 3 and q.SourceSystemQuoteReference = x.QuoteCodeReference
        left join ${ods.target}.Policy  p   on p.PolicyCode = x.PolicyCode
        where   x.rn = 1
        ) x

## Checks


In [ ]:
%%pyspark
# Checks for the tables this notebook built. All are checked and logged to OdsBuildLog; the cell then fails if
# any check failed, naming the table and the likely cause.
import json, re
from functools import reduce
from pyspark.sql import functions as F
SPEC = json.loads(r"""[{"table": "QuoteHome", "bk": ["SourceSystemId", "SourceSystemQuoteReference"], "sk": null, "nullable": [], "allow": []}, {"table": "PolicyHomeQsEvent", "bk": ["HomeRiskId"], "sk": null, "nullable": [], "allow": []}, {"table": "QuoteHomePrice", "bk": ["ResponseQuoteId"], "sk": null, "nullable": [], "allow": []}, {"table": "QuoteHomePaymentRequest", "bk": ["GlobalPaymentReference"], "sk": null, "nullable": [], "allow": []}, {"table": "QuoteHomePaymentResponse", "bk": ["PaymentResponseId"], "sk": null, "nullable": [], "allow": []}, {"table": "QuoteHomeSale", "bk": ["SaleRecordId"], "sk": null, "nullable": [], "allow": []}]""")
PII_NAME = re.compile(r"(e_?mail|forename|first_?name|surname|last_?name|full_?name|second_?name|phone|mobile|telephone|post_?code|eircode(?!_?area)|address_?line|risk_?address|postal_?address|dob$|date_?of_?birth|licen[cs]e_?num|registration_?no|reg_?number|registration_number|^ani$|^dnis$|originatingdnis|^payer_|iban|^bic|^contact$|rendered_?message)", re.I)
EMAIL_RE = r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"
MOBILE_RE = r"(^|[^0-9A-Za-z-])(\+?353 ?|0)8[35-9][ -]?[0-9]{3}[ -]?[0-9]{4}([^0-9A-Za-z-]|$)"   # stands alone, not inside an id
rows, failures = [], []
for s in SPEC:
    t = f"{target_schema}.{s['table']}"
    df = spark.table(t)
    bk = s["bk"]
    must = [c for c in bk if c not in s["nullable"]]
    agg = df.agg(F.count(F.lit(1)).alias("n"),
                 F.countDistinct(F.struct(*bk)).alias("d_bk"),
                 F.sum(F.when(reduce(lambda a, b: a | b, [F.col(c).isNull() for c in must] or [F.lit(False)]), 1).otherwise(0)).alias("null_bk"),
                 *( [F.countDistinct(s["sk"]).alias("d_sk")] if s["sk"] else [] )).first()
    pii_cols = [c for c in df.columns if PII_NAME.search(c) and not c.endswith("Key") and c not in s["allow"]]
    text_cols = [f.name for f in df.schema.fields if f.dataType.simpleString() == "string"]   # hashes never match
    hits = {}
    if text_cols:
        found = df.select(*text_cols).limit(1000000).agg(
            *[F.sum((F.col(c).rlike(EMAIL_RE) | F.col(c).rlike(MOBILE_RE)).cast("int")).alias(c) for c in text_cols]).first().asDict()
        hits = {c: v for c, v in found.items() if v}
    problems, advice = [], []
    keys = " + ".join(bk)
    if agg["n"] == 0:
        problems.append("empty")
        advice.append("the table is empty: its source is empty, or a filter (cut-off date, join) removed every row")
    if agg["d_bk"] != agg["n"]:
        problems.append(f"{agg['n'] - agg['d_bk']:,} duplicate business keys")
        advice.append(f"several rows share a key ({keys}): the source sent the same record more than once, or the "
                      f"dedupe rule in the cell no longer holds. Examples: select {', '.join(bk)}, count(*) from {t} "
                      f"group by {', '.join(bk)} having count(*) > 1")
    if agg["null_bk"]:
        problems.append(f"{agg['null_bk']:,} null business keys")
        advice.append(f"key columns ({', '.join(must)}) are empty on some rows: if that is legitimate for a column, add it "
                      f"to key_nullable for this table in the config; otherwise the source changed")
    if s["sk"] and agg["d_sk"] != agg["n"]:
        problems.append("surrogate key not unique")
        advice.append("the surrogate key repeats: the business key is not unique (see above)")
    if pii_cols:
        problems.append(f"PII-named columns {pii_cols}")
        advice.append("columns named like personal data: hash them (HASH_*) or leave them out in the config")
    if hits:
        problems.append(f"email/mobile values in {hits}")
        advice.append("emails / mobile numbers inside these columns: wrap the column in REDACT() (or hash it) in the config")
    rows.append((batch_id, s["table"], agg["n"], agg["n"] - agg["d_bk"], agg["null_bk"], "; ".join(problems) or "ok"))
    print(f"{s['table']:<28} {agg['n']:>13,}  {'; '.join(problems) or 'ok'}")
    if problems:
        failures.append(f"{s['table']}: " + " | ".join(advice))
log = spark.createDataFrame(rows, "OdsBatchId string, TableName string, RowCount long, DuplicateKeys long, NullKeys long, Result string")
log.withColumn("CheckedAt", F.current_timestamp()).write.mode("append").saveAsTable(f"{target_schema}.OdsBuildLog")
if failures:
    ods_run_log("22_Quote_Home", "failed", "; ".join(f.split(":")[0] for f in failures))
    raise AssertionError("ods checks failed:\n- " + "\n- ".join(failures))
ods_run_log("22_Quote_Home", "succeeded")
